In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, mean_absolute_error, mean_squared_error, r2_score, roc_auc_score
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)

from tqdm import tqdm
import copy
import json 
import time
import os

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Torch: 2.5.1+cu121
CUDA available: True
GPU: NVIDIA A100-SXM4-40GB


In [2]:
df = pd.read_parquet("Data/dipser_dataset.parquet", engine="pyarrow")
df["age_group"] = pd.cut(df["age"], bins=[13, 20, 22, 26, 44]) # bins=[13, 17, 20, 22, 26, 44]
df.head()

,group,time,time_sec,image_path,metadata,subject,experiment,self_labeling emotion,self_labeling attention,labeler_02 attention,labeler_04 attention,labeler_02 emotion,labeler_01 attention,labeler_03 emotion,labeler_03 attention,labeler_01 emotion,labeler_04 emotion,self_labeling emotionfilled,self_labeling attentionfilled,labeler_02 attentionfilled,labeler_04 attentionfilled,labeler_02 emotionfilled,labeler_01 attentionfilled,labeler_03 emotionfilled,labeler_03 attentionfilled,labeler_01 emotionfilled,labeler_04 emotionfilled,samsung_rotation_vector value0_mean,samsung_rotation_vector value0_std,samsung_rotation_vector value1_mean,samsung_rotation_vector value1_std,samsung_rotation_vector value2_mean,samsung_rotation_vector value2_std,samsung_rotation_vector value3_mean,samsung_rotation_vector value3_std,samsung_rotation_vector value4_mean,samsung_rotation_vector value4_std,lsm6dso_gyroscope value0_mean,lsm6dso_gyroscope value0_std,lsm6dso_gyroscope value1_mean,lsm6dso_gyroscope value1_std,lsm6dso_gyroscope value2_mean,lsm6dso_gyroscope value2_std,samsung_linear_acceleration_sensor value0_mean,samsung_linear_acceleration_sensor value0_std,samsung_linear_acceleration_sensor value1_mean,samsung_linear_acceleration_sensor value1_std,samsung_linear_acceleration_sensor value2_mean,samsung_linear_acceleration_sensor value2_std,opt3007_light value0_mean,opt3007_light value0_std,heart_rate,heart_rate_std,accel_magnitude_mean,accel_magnitude_std,gyro_magnitude_mean,gyro_magnitude_std,labeler_05 emotion,labeler_05 attention,labeler_05 emotionfilled,labeler_05 attentionfilled,invalid_reason,gender,age,race,subject_experiment_id,attention,subject_id,age_group
0,group01,2026-05-08 10:40:43.047895,0,/home/cfragkiadakis/prjs2039/DIPSER/group01/experiment01/subject_01/images/10_40_43_047895.png,/home/cfragkiadakis/prjs2039/DIPSER/group01/experiment01/subject_01/metadata/10_40_43_047895.json,subject_01,experiment01,6.0,3.0,4.0,3.0,8.0,3.0,9.0,3.0,1.0,7.0,6.0,3.0,4.0,3.0,8.0,3.0,9.0,3.0,1.0,7.0,-0.109524,0.000163,-0.659717,0.000588,0.632875,0.000495,0.390186,0.000218,246.0,0.0,-0.006022,0.006482,-0.003838,0.003665,-0.000864,0.006845,0.052984,0.034713,-0.132399,0.047077,0.017549,0.043508,63.0,0.0,78.0,NaN,0.155556,0.041540,0.011095,0.005498,NaN,NaN,NaN,NaN,valid,male,26.0,white,group01_experiment01_subject_01,3.25,group01_subject_01,"(22, 26]"
1,group01,2026-05-08 10:40:43.195317,0,/home/cfragkiadakis/prjs2039/DIPSER/group01/experiment01/subject_01/images/10_40_43_195317.png,/home/cfragkiadakis/prjs2039/DIPSER/group01/experiment01/subject_01/metadata/10_40_43_195317.json,subject_01,experiment01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.0,3.0,4.0,3.0,8.0,3.0,9.0,3.0,1.0,7.0,-0.109485,0.000109,-0.660015,0.000520,0.632620,0.000426,0.390106,0.000225,246.0,0.0,-0.004508,0.005235,-0.004068,0.003267,-0.001026,0.005944,0.053056,0.036979,-0.124020,0.045047,0.012282,0.043478,63.0,0.0,78.0,NaN,0.148634,0.038779,0.009572,0.004378,NaN,NaN,NaN,NaN,valid,male,26.0,white,group01_experiment01_subject_01,3.25,group01_subject_01,"(22, 26]"
2,group01,2026-05-08 10:40:43.295405,0,/home/cfragkiadakis/prjs2039/DIPSER/group01/experiment01/subject_01/images/10_40_43_295405.png,/home/cfragkiadakis/prjs2039/DIPSER/group01/experiment01/subject_01/metadata/10_40_43_295405.json,subject_01,experiment01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.0,3.0,4.0,3.0,8.0,3.0,9.0,3.0,1.0,7.0,-0.109478,0.000101,-0.660194,0.000514,0.632473,0.000406,0.390043,0.000252,246.0,0.0,-0.004581,0.005193,-0.003897,0.003153,-0.000525,0.005644,0.054420,0.036670,-0.129478,0.044805,0.015730,0.040968,63.0,0.0,78.0,NaN,0.153129,0.038995,0.009384,0.004079,NaN,NaN,NaN,NaN,valid,male,26.0,white,group01_experiment01_subject_01,3.25,group01_subject_01,"(22, 26]"
3,group01,2026-05-08 10:40:43.395835,0,/home/cfragkiadakis/prjs2039/DIPSER/group01/experiment01/subject_01/images/10_40_43_395835.png,/home/cfragkiadakis/prjs2039/DIPSER/group01/experiment01/subject_01/metadata/10_40_43_395835.json,subject_01,experime

In [3]:
df[["attention", "image_path", "age", "gender"]].isna().sum()

attention     0
image_path    0
age           0
gender        0
dtype: int64

In [4]:
SEED = 42
SEQUENCE_LENGTH = 10
MAX_MISSING_VISUAL_FRAMES = SEQUENCE_LENGTH - 1 # remove a sequence only when all the images are unavailable

# Label-imbalance intervention toggles.
USE_WEIGHTED_LOSS = True
USE_WEIGHTED_SAMPLER = False
WEIGHT_ALPHA = 0.5     # 0=no weighting, 1=full inverse-frequency weighting
# ATTENTION_BINS = [2, 2.5, 3, 3.5, 4, 4.75]
ATTENTION_BINS = [2, 2.5, 3, 3.5, 4.75]

STRATIFY_COLUMN = 'age_group'
BATCH_SIZE = 32
NUM_WORKERS = 8

In [5]:
# Keep one frame per second, matching the cached feature setup used in Fusion.
df_sec = (
    df.sort_values(["subject_experiment_id", "time_sec"])
    .groupby(["subject_experiment_id", "time_sec"])
    .first()
    .reset_index())

df_sec.shape

(111754, 69)

In [7]:
reconstructed_sequences = []

for sequence_id, sequence_df in df_sec.groupby("subject_experiment_id"):
    sequence_df = sequence_df.sort_values("time_sec")

    first_second = sequence_df["time_sec"].min()
    last_second = sequence_df["time_sec"].max()

    complete_seconds = pd.DataFrame({
        "time_sec": np.arange(first_second, last_second + 1)
    })

    reconstructed = complete_seconds.merge(
        sequence_df,
        on="time_sec",
        how="left"
    )

    reconstructed["subject_experiment_id"] = sequence_id
    reconstructed["missing_visual_input"] = reconstructed["image_path"].isna().astype(int)
    reconstructed_sequences.append(reconstructed)

temporal_frame_dataset = pd.concat(reconstructed_sequences, ignore_index=True)

In [8]:
temporal_frame_dataset.shape

(121332, 70)

In [9]:
feature_index = pd.read_parquet('Data/clip_vitl14_features/clip_vitl14_frame_index.parquet')
feature_store_path =  'Data/clip_vitl14_features/clip_vitl14_frame_features.npy'
VISUAL_FEATURE_DIM = 768


feature_row_by_path = dict(zip(feature_index["image_path"], feature_index["feature_row"]))
CNN_FEATURES = feature_store_path.split('/')[-2]

temporal_frame_dataset["feature_row"] = (
    temporal_frame_dataset["image_path"]
    .map(feature_row_by_path)
    .fillna(-1)
    .astype(np.int64))

missing_feature_rows = (
    (temporal_frame_dataset["missing_visual_input"] == 0)
    & (temporal_frame_dataset["feature_row"] == -1)
).sum()

print("Non-missing image rows without cached features:", missing_feature_rows)
temporal_frame_dataset.head()

Non-missing image rows without cached features: 0


,time_sec,subject_experiment_id,group,time,image_path,metadata,subject,experiment,self_labeling emotion,self_labeling attention,labeler_02 attention,labeler_04 attention,labeler_02 emotion,labeler_01 attention,labeler_03 emotion,labeler_03 attention,labeler_01 emotion,labeler_04 emotion,self_labeling emotionfilled,self_labeling attentionfilled,labeler_02 attentionfilled,labeler_04 attentionfilled,labeler_02 emotionfilled,labeler_01 attentionfilled,labeler_03 emotionfilled,labeler_03 attentionfilled,labeler_01 emotionfilled,labeler_04 emotionfilled,samsung_rotation_vector value0_mean,samsung_rotation_vector value0_std,samsung_rotation_vector value1_mean,samsung_rotation_vector value1_std,samsung_rotation_vector value2_mean,samsung_rotation_vector value2_std,samsung_rotation_vector value3_mean,samsung_rotation_vector value3_std,samsung_rotation_vector value4_mean,samsung_rotation_vector value4_std,lsm6dso_gyroscope value0_mean,lsm6dso_gyroscope value0_std,lsm6dso_gyroscope value1_mean,lsm6dso_gyroscope value1_std,lsm6dso_gyroscope value2_mean,lsm6dso_gyroscope value2_std,samsung_linear_acceleration_sensor value0_mean,samsung_linear_acceleration_sensor value0_std,samsung_linear_acceleration_sensor value1_mean,samsung_linear_acceleration_sensor value1_std,samsung_linear_acceleration_sensor value2_mean,samsung_linear_acceleration_sensor value2_std,opt3007_light value0_mean,opt3007_light value0_std,heart_rate,heart_rate_std,accel_magnitude_mean,accel_magnitude_std,gyro_magnitude_mean,gyro_magnitude_std,labeler_05 emotion,labeler_05 attention,labeler_05 emotionfilled,labeler_05 attentionfilled,invalid_reason,gender,age,race,attention,subject_id,age_group,missing_visual_input,feature_row
0,0,group01_experiment01_subject_01,group01,2026-05-08 10:40:43.047895,/home/cfragkiadakis/prjs2039/DIPSER/group01/experiment01/subject_01/images/10_40_43_047895.png,/home/cfragkiadakis/prjs2039/DIPSER/group01/experiment01/subject_01/metadata/10_40_43_047895.json,subject_01,experiment01,6.0,3.0,4.0,3.0,8.0,3.0,9.0,3.0,1.0,7.0,6.0,3.0,4.0,3.0,8.0,3.0,9.0,3.0,1.0,7.0,-0.109524,0.000163,-0.659717,0.000588,0.632875,0.000495,0.390186,0.000218,246.0,0.0,-0.006022,0.006482,-0.003838,0.003665,-0.000864,0.006845,0.052984,0.034713,-0.132399,0.047077,0.017549,0.043508,63.0,0.0,78.0,NaN,0.155556,0.041540,0.011095,0.005498,NaN,NaN,NaN,NaN,valid,male,26.0,white,3.25,group01_subject_01,"(22, 26]",0,0
1,1,group01_experiment01_subject_01,group01,2026-05-08 10:40:44.195980,/home/cfragkiadakis/prjs2039/DIPSER/group01/experiment01/subject_01/images/10_40_44_195980.png,/home/cfragkiadakis/prjs2039/DIPSER/group01/experiment01/subject_01/metadata/10_40_44_195980.json,subject_01,experiment01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.0,3.0,3.0,3.0,1.0,3.0,9.0,3.0,1.0,7.0,-0.109484,0.000186,-0.663393,0.001822,0.629769,0.001544,0.388982,0.000659,246.0,0.0,-0.017314,0.012752,-0.005738,0.003119,-0.002246,0.006452,0.055115,0.038306,-0.129263,0.050346,0.014268,0.035097,63.0,0.0,77.0,NaN,0.152183,0.044628,0.020409,0.011592,NaN,NaN,NaN,NaN,valid,male,26.0,white,3.00,group01_subject_01,"(22, 26]",0,1
2,2,group01_experiment01_subject_01,group01,2026-05-08 10:40:45.095651,/home/cfragkiadakis/prjs2039/DIPSER/group01/experiment01/subject_01/images/10_40_45_095651.png,/home/cfragkiadakis/prjs2039/DIPSER/group01/experiment01/subject_01/metadata/10_40_45_095651.json,subject_01,experiment01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.0,3.0,3.0,3.0,1.0,3.0,9.0,3.0,1.0,7.0,-0.109878,0.000156,-0.667409,0.000379,0.626474,0.000320,0.387325,0.000218,246.0,0.0,-0.003115,0.007570,-0.002407,0.002732,-0.001051,0.007162,0.053606,0.034863,-0.129119,0.055241,0.013599,0.039643,63.0,0.0,77.0,NaN,0.152138,0.048870,0.010279,0.005109,NaN,NaN,NaN,NaN,valid,male,26.0,white,3.00,group01_subject_01,"(22, 26]",0,2
3,3,group01_experiment01_subject_01,group01,2026-05-08 10:40:46.641389,/home/cfragkiadakis/prjs2039/DIPSER/group01/experiment01/subject_01/images/10_40_46_641389.png,/home/cfragkiadakis/prjs2039/DIPSER/grou

In [10]:
def create_temporal_sequences(frame_df, sequence_length=SEQUENCE_LENGTH):
    sequences = []

    for sequence_id, sequence_df in frame_df.groupby("subject_experiment_id"):
        sequence_df = sequence_df.sort_values("time_sec").reset_index(drop=True)

        if len(sequence_df) < sequence_length:
            continue

        for i in range(sequence_length - 1, len(sequence_df)):
            target = sequence_df.iloc[i]["attention"]

            if pd.isna(target):
                continue

            history = sequence_df.iloc[i - sequence_length + 1:i + 1]
            missing_flags = history["missing_visual_input"].astype(np.float32).values

            if missing_flags.sum() > MAX_MISSING_VISUAL_FRAMES:
                continue

            sequences.append({
                "subject_experiment_id": sequence_id,
                "time_sec": int(sequence_df.iloc[i]["time_sec"]),
                "feature_rows": history["feature_row"].tolist(),
                "missing_flags": missing_flags.tolist(),
                "target": float(target),
            })

    return pd.DataFrame(sequences)

temporal_sequence_df = create_temporal_sequences(temporal_frame_dataset)
temporal_sequence_df.shape

(108033, 5)

In [11]:
subject_info = df[["subject_experiment_id", "subject_id", "age", "gender", "age_group"]].drop_duplicates()

# append metadata columns
temporal_sequence_df = temporal_sequence_df.merge(
    subject_info,
    on="subject_experiment_id",
    how="left")

temporal_sequence_df[["subject_experiment_id", "subject_id", "gender", "age_group", "target"]].head()

,subject_experiment_id,subject_id,gender,age_group,target
0,group01_experiment01_subject_01,group01_subject_01,male,"(22, 26]",3.00
1,group01_experiment01_subject_01,group01_subject_01,male,"(22, 26]",3.00
2,group01_experiment01_subject_01,group01_subject_01,male,"(22, 26]",3.00
3,group01_experiment01_subject_01,group01_subject_01,male,"(22, 26]",3.00
4,group01_experiment01_subject_01,group01_subject_01,male,"(22, 26]",3.25


In [12]:
# Select one fixed age-stratified split using demographic metadata only.
# Model predictions, labels, and test performance are never used to choose it.
SPLIT_SEARCH_TRIALS = 1_000
MIN_MALE_VAL_SUBJECTS = 3
MIN_MALE_TEST_SUBJECTS = 3


def demographic_distance(split_df, full_df, column):
    categories = sorted(full_df[column].astype(str).unique())
    full_dist = full_df[column].astype(str).value_counts(normalize=True).reindex(categories, fill_value=0.0)
    split_dist = split_df[column].astype(str).value_counts(normalize=True).reindex(categories, fill_value=0.0)
    return float(np.abs(split_dist - full_dist).sum())


def find_constrained_age_stratified_split(subject_df, trials=SPLIT_SEARCH_TRIALS, seed=SEED):
    candidates = []
    all_age_groups = set(subject_df["age_group"].astype(str))

    for offset in range(trials):
        split_seed = seed + offset
        try:
            train_sub, temp_sub = train_test_split(
                subject_df, test_size=0.3, stratify=subject_df["age_group"], random_state=split_seed
            )
            val_sub, test_sub = train_test_split(
                temp_sub, test_size=0.5, stratify=temp_sub["age_group"], random_state=split_seed
            )
        except ValueError:
            continue

        val_males = int((val_sub["gender"].astype(str).str.lower() == "male").sum())
        test_males = int((test_sub["gender"].astype(str).str.lower() == "male").sum())
        penalty = (
            max(0, MIN_MALE_VAL_SUBJECTS - val_males) * 100
            + max(0, MIN_MALE_TEST_SUBJECTS - test_males) * 100
            + (0 if set(val_sub["age_group"].astype(str)) == all_age_groups else 100)
            + (0 if set(test_sub["age_group"].astype(str)) == all_age_groups else 100)
        )
        balance_score = sum(
            demographic_distance(split, subject_df, column)
            for split in [train_sub, val_sub, test_sub]
            for column in ["age_group", "gender"]
        )
        candidates.append((penalty, balance_score, split_seed, train_sub.copy(), val_sub.copy(), test_sub.copy()))

    if not candidates:
        raise RuntimeError("Could not construct an age-stratified subject split.")

    best = min(candidates, key=lambda item: (item[0], item[1], item[2]))
    if best[0] > 0:
        print("WARNING: No split satisfied every requested representation constraint.")
    return best[3], best[4], best[5], best[2], best[0], best[1]


subject_df = (
    temporal_sequence_df[["subject_id", "age_group", "gender"]]
    .dropna(subset=["subject_id", "age_group", "gender"])
    .drop_duplicates("subject_id")
    .copy()
)
train_sub, val_sub, test_sub, SPLIT_RANDOM_STATE, split_penalty, split_balance_score = (
    find_constrained_age_stratified_split(subject_df)
)

train_df = temporal_sequence_df[temporal_sequence_df["subject_id"].isin(train_sub["subject_id"])].copy()
val_df = temporal_sequence_df[temporal_sequence_df["subject_id"].isin(val_sub["subject_id"])].copy()
test_df = temporal_sequence_df[temporal_sequence_df["subject_id"].isin(test_sub["subject_id"])].copy()


def split_demographic_table(split_df, split_name):
    rows = []
    for attribute in ["age_group", "gender"]:
        for group, count in split_df[attribute].astype(str).value_counts().sort_index().items():
            rows.append({
                "split": split_name,
                "attribute": attribute,
                "group": group,
                "subjects": int(count),
                "proportion": float(count / len(split_df)),
            })
    return pd.DataFrame(rows)


split_demographics = pd.concat(
    [
        split_demographic_table(train_sub, "train"),
        split_demographic_table(val_sub, "validation"),
        split_demographic_table(test_sub, "test"),
    ],
    ignore_index=True,
)

print(f"Selected demographic-only split random state: {SPLIT_RANDOM_STATE}")
print(f"Constraint penalty: {split_penalty}; demographic balance score: {split_balance_score:.4f}")
display(split_demographics)

pd.DataFrame({
    "split": ["train", "val", "test"],
    "sequences": [len(train_df), len(val_df), len(test_df)],
    "subjects": [train_df.subject_id.nunique(), val_df.subject_id.nunique(), test_df.subject_id.nunique()],
    "target_mean": [train_df.target.mean(), val_df.target.mean(), test_df.target.mean()],
    "visual_missing_rate": [
        train_df.missing_flags.apply(np.mean).mean(),
        val_df.missing_flags.apply(np.mean).mean(),
        test_df.missing_flags.apply(np.mean).mean(),
    ],
})


Selected demographic-only split random state: 45
Constraint penalty: 0; demographic balance score: 0.3941


,split,attribute,group,subjects,proportion
0,train,age_group,"(13, 20]",13,0.333333
1,train,age_group,"(20, 22]",10,0.256410
2,train,age_group,"(22, 26]",10,0.256410
3,train,age_group,"(26, 44]",6,0.153846
4,train,gender,female,27,0.692308
5,train,gender,male,12,0.307692
6,validation,age_group,"(13, 20]",3,0.333333
7,validation,age_group,"(20, 22]",3,0.333333
8,validation,age_group,"(22, 26]",2,0.222222
9,validation,age_group,"(26, 44]",1,0.111111


,split,sequences,subjects,target_mean,visual_missing_rate
0,train,75189,39,2.971392,0.063306
1,val,17208,9,2.948236,0.057438
2,test,15636,9,2.895210,0.052315


In [13]:
# Quantify target imbalance and create inverse-frequency train weights.
# Weights are learned from train only; val/test weights are diagnostic only.
def add_attention_bin_weights(alpha, train_df, val_df, test_df):
    train_df = train_df.copy()
    val_df = val_df.copy()
    test_df = test_df.copy()

    train_bins = pd.cut(train_df["target"], bins=ATTENTION_BINS, include_lowest=True)
    bin_counts = train_bins.value_counts().sort_index()
    nonzero_counts = bin_counts[bin_counts > 0]

    bin_weights = (len(train_df) / (len(nonzero_counts) * nonzero_counts)) ** alpha

    for split_df in [train_df, val_df, test_df]:
        split_bins = pd.cut(split_df["target"], bins=ATTENTION_BINS, include_lowest=True)
        split_df["attention_bin"] = split_bins.astype(str)
        split_df["sample_weight"] = split_bins.map(bin_weights).astype(float).fillna(1.0)

    train_weight_mean = train_df["sample_weight"].mean()

    for split_df in [train_df, val_df, test_df]:
        split_df["sample_weight"] = split_df["sample_weight"] / train_weight_mean

    normalized_bin_weights = bin_weights / train_weight_mean

    imbalance_table = pd.DataFrame({
        "bin": bin_counts.index.astype(str),
        "train_count": bin_counts.values,
        "weight": [
            float(normalized_bin_weights.get(idx, np.nan))
            for idx in bin_counts.index
        ],
    })

    return train_df, val_df, test_df, imbalance_table

train_df, val_df, test_df, imbalance_table = add_attention_bin_weights(WEIGHT_ALPHA, train_df, val_df, test_df)
display(imbalance_table)

pd.DataFrame({
    "split": ["train", "val", "test"],
    "sequences": [len(train_df), len(val_df), len(test_df)],
    "subjects": [train_df.subject_id.nunique(), val_df.subject_id.nunique(), test_df.subject_id.nunique()],
    "target_mean": [train_df.target.mean(), val_df.target.mean(), test_df.target.mean()],
    "mean_sample_weight": [train_df.sample_weight.mean(), val_df.sample_weight.mean(), test_df.sample_weight.mean()],
})

,bin,train_count,weight
0,"(1.999, 2.5]",15291,1.160744
1,"(2.5, 3.0]",35590,0.760835
2,"(3.0, 3.5]",18734,1.048671
3,"(3.5, 4.75]",5574,1.922521


,split,sequences,subjects,target_mean,mean_sample_weight
0,train,75189,39,2.971392,1.000000
1,val,17208,9,2.948236,0.981307
2,test,15636,9,2.895210,0.968726


In [14]:
train_df.shape, val_df.shape, test_df.shape

((75189, 11), (17208, 11), (15636, 11))

In [15]:
len(set(train_df['subject_experiment_id'])), len(set(val_df['subject_experiment_id'])) ,len(set(test_df['subject_experiment_id']))

(285, 64, 58)

In [16]:
class TemporalFeatureDataset(Dataset):

    def __init__(self, sequence_df, feature_store_path):
        self.df = sequence_df.reset_index(drop=True)
        self.feature_store_path = feature_store_path
        self.feature_store = None

    def __len__(self):
        return len(self.df)

    def _features(self):
        if self.feature_store is None:
            self.feature_store = np.load(self.feature_store_path, mmap_mode="r")
        return self.feature_store

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        feature_rows = np.asarray(row["feature_rows"], dtype=np.int64)
        visual_features = np.zeros((len(feature_rows), VISUAL_FEATURE_DIM), dtype=np.float32)

        valid = feature_rows >= 0
        visual_features[valid] = self._features()[feature_rows[valid]]

        visual_features = torch.tensor(visual_features, dtype=torch.float32)
        missing_flags = torch.tensor(row["missing_flags"], dtype=torch.float32)
        target = torch.tensor(row["target"], dtype=torch.float32)
        sample_weight = torch.tensor(row["sample_weight"], dtype=torch.float32)

        return visual_features, missing_flags, target, sample_weight, idx

In [17]:
train_dataset = TemporalFeatureDataset(train_df, feature_store_path)
val_dataset = TemporalFeatureDataset(val_df, feature_store_path)
test_dataset = TemporalFeatureDataset(test_df, feature_store_path)

loader_kwargs = {
    "batch_size": BATCH_SIZE,
    "num_workers": NUM_WORKERS,
    "pin_memory": torch.cuda.is_available(),
    "persistent_workers": NUM_WORKERS > 0,
    "prefetch_factor": 4 if NUM_WORKERS > 0 else None,
}
loader_kwargs = {k: v for k, v in loader_kwargs.items() if v is not None}

if USE_WEIGHTED_SAMPLER:
    sampler = torch.utils.data.WeightedRandomSampler(
        weights=torch.tensor(train_df["sample_weight"].values, dtype=torch.double),
        num_samples=len(train_df),
        replacement=True
    )
    train_loader = DataLoader(train_dataset, sampler=sampler, **loader_kwargs)
else:
    train_loader = DataLoader(train_dataset, shuffle=True, **loader_kwargs)

val_loader = DataLoader(val_dataset, shuffle=False, **loader_kwargs)
test_loader = DataLoader(test_dataset, shuffle=False, **loader_kwargs)

### Training

In [18]:
class Feature_GRU(nn.Module):

    def __init__(self, feature_dim=VISUAL_FEATURE_DIM, hidden_size=128, dropout=0.3):
        super().__init__()

        self.feature_projection = nn.Sequential(
            nn.Linear(feature_dim + 1, 512),
            nn.LayerNorm(512),
            nn.ReLU(),
            nn.Dropout(0.2),
        )

        self.gru = nn.GRU(
            input_size=512,
            hidden_size=hidden_size,
            num_layers=1,
            batch_first=True
        )

        self.temporal_norm = nn.LayerNorm(hidden_size)

        self.regressor = nn.Sequential(
            nn.Linear(hidden_size, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1)
        )

    def forward(self, visual_features, missing_flags):
        missing_flags = missing_flags.unsqueeze(-1)
        features = torch.cat([visual_features, missing_flags], dim=-1)
        features = self.feature_projection(features)

        output, _ = self.gru(features)
       # temporal_features = output.mean(dim=1)
        temporal_features = output[:, -1, :]
        temporal_features = self.temporal_norm(temporal_features)

        prediction = self.regressor(temporal_features)
        return prediction.squeeze(1)

### Fixed-split multi-seed visual baseline stability

This notebook measures the unregularized Temporal Visual model across ten
training seeds on one fixed constrained age-stratified split. Validation and
test each contain six female and three male subjects, with every age group
represented.

The visual GRU architecture, weighted MSE objective, and shuffled training
loader are preserved from `Temporal Visual.ipynb`.


In [19]:
import copy
import random
from importlib import reload
import src.evaluation as ev

ev = reload(ev)


def set_global_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def make_model():
    return Feature_GRU()


def make_train_loader():
    loader_kwargs = {
        "batch_size": BATCH_SIZE,
        "num_workers": NUM_WORKERS,
        "pin_memory": torch.cuda.is_available(),
        "persistent_workers": NUM_WORKERS > 0,
        "prefetch_factor": 4 if NUM_WORKERS > 0 else None,
    }
    loader_kwargs = {key: value for key, value in loader_kwargs.items() if value is not None}
    if USE_WEIGHTED_SAMPLER:
        sampler = torch.utils.data.WeightedRandomSampler(
            weights=torch.tensor(train_df["sample_weight"].values, dtype=torch.double),
            num_samples=len(train_df),
            replacement=True,
        )
        return DataLoader(train_dataset, sampler=sampler, **loader_kwargs)
    return DataLoader(train_dataset, shuffle=True, **loader_kwargs)


def weighted_task_loss(per_sample_loss, sample_weights):
    if USE_WEIGHTED_LOSS:
        return (per_sample_loss * sample_weights).sum() / sample_weights.sum().clamp_min(1e-8)
    return per_sample_loss.mean()


def train_one_epoch_baseline(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0.0
    preds_all, labels_all = [], []
    for visual_features, missing_flags, labels, sample_weights, _idx in tqdm(loader, desc="Training", leave=False):
        visual_features = visual_features.to(device)
        missing_flags = missing_flags.to(device)
        labels = labels.to(device)
        sample_weights = sample_weights.to(device)

        preds = model(visual_features, missing_flags)
        loss = weighted_task_loss(criterion(preds, labels), sample_weights)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += float(loss.item())
        preds_all.extend(preds.detach().cpu().numpy())
        labels_all.extend(labels.detach().cpu().numpy())

    return {
        "loss": total_loss / len(loader),
        "mae": float(mean_absolute_error(labels_all, preds_all)),
        "rmse": float(np.sqrt(mean_squared_error(labels_all, preds_all))),
    }


def evaluate_loader(model, loader, sequence_df):
    model.eval()
    preds_all, labels_all, indices_all = [], [], []
    with torch.no_grad():
        for visual_features, missing_flags, labels, _sample_weights, idx in tqdm(loader, desc="Evaluating", leave=False):
            preds = model(visual_features.to(device), missing_flags.to(device))
            preds_all.extend(preds.detach().cpu().numpy())
            labels_all.extend(labels.detach().cpu().numpy())
            indices_all.extend(idx.detach().cpu().numpy())

    frame = sequence_df.iloc[np.asarray(indices_all, dtype=int)][
        ["subject_experiment_id", "subject_id", "time_sec", "attention_bin", "gender", "age", "age_group"]
    ].reset_index(drop=True)
    frame.insert(0, "true", np.asarray(labels_all, dtype=float))
    frame.insert(0, "pred", np.asarray(preds_all, dtype=float))

    overall = ev.compute_prediction_metrics(frame)
    age_mae, age_worst, age_gap = ev.compute_group_mae(frame, "age_group")
    gender_mae, gender_worst, gender_gap = ev.compute_group_mae(frame, "gender")
    return {
        "mae": overall["mae"],
        "rmse": overall["rmse"],
        "r2": overall["r2"],
        "true_mean": overall["true_mean"],
        "pred_mean": overall["pred_mean"],
        "age_worst_group_mae": age_worst,
        "age_gap": age_gap,
        "gender_worst_group_mae": gender_worst,
        "gender_gap": gender_gap,
        "age_mae_per_group": age_mae.to_dict(),
        "gender_mae_per_group": gender_mae.to_dict(),
    }, frame


In [20]:
class EarlyStopping:
    def __init__(self, patience, model_path):
        self.patience = patience
        self.model_path = model_path
        self.best_score = float("inf")
        self.best_epoch = None
        self.counter = 0

    def step(self, score, model, epoch):
        if score < self.best_score:
            self.best_score = float(score)
            self.best_epoch = epoch
            self.counter = 0
            torch.save(copy.deepcopy(model.state_dict()), self.model_path)
        else:
            self.counter += 1
        return self.counter >= self.patience


LOSS_TYPE = "mse"
criterion = nn.MSELoss(reduction="none")
EARLY_STOPPING_METRIC = "rmse"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
NUM_EPOCHS = 50
PATIENCE = 7

RUN_SEEDS = [42, 100, 2000, 2025, 2026, 2027, 2048, 4096, 7000, 8192]

RESULTS_DIR = "results/Temporal Visual Baseline Stability"
MODEL_DIR = "models/Temporal Visual Baseline Stability"
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

print(f"Fixed split random state: {SPLIT_RANDOM_STATE}")
print(f"Training batches shuffled: True")
print(f"Training seeds: {RUN_SEEDS}")

Fixed split random state: 45
Training batches shuffled: True
Training seeds: [42, 100, 2000, 2025, 2026, 2027, 2048, 4096, 7000, 8192]


In [21]:
run_records = []
test_predictions = {}

for run_seed in RUN_SEEDS:
    set_global_seed(run_seed)
    run_name = f"visual_baseline_fixed_age_split_seed{run_seed}"
    model_path = os.path.join(MODEL_DIR, f"{run_name}.pt")
    model = make_model().to(device)
    optimizer = torch.optim.Adam(
        filter(lambda parameter: parameter.requires_grad, model.parameters()),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.3, patience=3)
    early_stopping = EarlyStopping(PATIENCE, model_path)
    run_train_loader = make_train_loader()
    history = []

    print(f"\n=== {run_name} ===")
    for epoch in range(NUM_EPOCHS):
        train_metrics = train_one_epoch_baseline(model, run_train_loader, optimizer, criterion)
        val_metrics, _ = evaluate_loader(model, val_loader, val_df)
        monitor = val_metrics[EARLY_STOPPING_METRIC]
        scheduler.step(monitor)
        history.append({
            "epoch": epoch + 1,
            "train_loss": train_metrics["loss"],
            "train_mae": train_metrics["mae"],
            "train_rmse": train_metrics["rmse"],
            "val_mae": val_metrics["mae"],
            "val_rmse": val_metrics["rmse"],
            "val_r2": val_metrics["r2"],
        })
        print(
            f"Epoch {epoch + 1:02d} | train MAE {train_metrics['mae']:.4f} | "
            f"val MAE {val_metrics['mae']:.4f} | val RMSE {val_metrics['rmse']:.4f}"
        )
        if early_stopping.step(monitor, model, epoch):
            print("Early stopping triggered")
            break

    model.load_state_dict(torch.load(model_path, map_location=device, weights_only=True))
    val_metrics, val_predictions = evaluate_loader(model, val_loader, val_df)
    test_metrics, test_frame = evaluate_loader(model, test_loader, test_df)
    test_predictions[run_name] = test_frame

    val_path = os.path.join(RESULTS_DIR, f"{run_name}_val_predictions.csv")
    test_path = os.path.join(RESULTS_DIR, f"{run_name}_test_predictions.csv")
    ev.save_prediction_frame(val_predictions, val_path)
    ev.save_prediction_frame(test_frame, test_path)
    run_records.append({
        "run_name": run_name,
        "run_seed": run_seed,
        "best_epoch": early_stopping.best_epoch + 1,
        "num_epochs_run": len(history),
        "model_path": model_path,
        "val_prediction_path": val_path,
        "test_prediction_path": test_path,
        "val_metrics": val_metrics,
        "test_metrics": test_metrics,
        "history": history,
    })

print("Finished all visual baseline runs.")



=== visual_baseline_fixed_age_split_seed42 ===


Epoch 01 | train MAE 0.3959 | val MAE 0.3040 | val RMSE 0.3963


Epoch 02 | train MAE 0.3465 | val MAE 0.2852 | val RMSE 0.3823


Epoch 03 | train MAE 0.3287 | val MAE 0.2911 | val RMSE 0.3813


Epoch 04 | train MAE 0.3125 | val MAE 0.2917 | val RMSE 0.3836


...

Epoch 10 | train MAE 0.2278 | val MAE 0.2948 | val RMSE 0.3920


Epoch 11 | train MAE 0.2143 | val MAE 0.2863 | val RMSE 0.3869


Epoch 12 | train MAE 0.2096 | val MAE 0.2874 | val RMSE 0.3864


Epoch 13 | train MAE 0.2068 | val MAE 0.2950 | val RMSE 0.3982
Early stopping triggered


Finished all visual baseline runs.


In [22]:
rows = []
for run in run_records:
    row = {
        "run_name": run["run_name"],
        "run_seed": run["run_seed"],
        "best_epoch": run["best_epoch"],
        "num_epochs_run": run["num_epochs_run"],
    }
    row.update({f"val_{key}": value for key, value in run["val_metrics"].items() if not isinstance(value, dict)})
    row.update({f"test_{key}": value for key, value in run["test_metrics"].items() if not isinstance(value, dict)})
    rows.append(row)

seed_results = pd.DataFrame(rows)
summary_metrics = [
    "test_mae", "test_rmse", "test_r2", "test_age_worst_group_mae",
    "test_age_gap", "test_gender_worst_group_mae", "test_gender_gap",
]
seed_summary = seed_results[summary_metrics].agg(["mean", "std", "min", "median", "max"]).T.reset_index(names="metric")
display(seed_results.round(4))
display(seed_summary.round(4))


,run_name,run_seed,best_epoch,num_epochs_run,val_mae,val_rmse,val_r2,val_true_mean,val_pred_mean,val_age_worst_group_mae,val_age_gap,val_gender_worst_group_mae,val_gender_gap,test_mae,test_rmse,test_r2,test_true_mean,test_pred_mean,test_age_worst_group_mae,test_age_gap,test_gender_worst_group_mae,test_gender_gap
0,visual_baseline_fixed_age_split_seed42,42,7,14,0.2845,0.3805,0.1815,2.9482,2.9367,0.3505,0.1134,0.2854,0.0026,0.2838,0.3613,0.1202,2.8952,2.8524,0.3415,0.1354,0.2914,0.0219
1,visual_baseline_fixed_age_split_seed100,100,1,8,0.2974,0.3864,0.1561,2.9482,2.9534,0.3583,0.1093,0.3112,0.0375,0.2745,0.3564,0.1442,2.8952,2.9141,0.3092,0.1076,0.2802,0.0166
2,visual_baseline_fixed_age_split_seed2000,2000,2,9,0.2887,0.3801,0.1832,2.9482,2.9490,0.3306,0.0818,0.2942,0.0150,0.2949,0.3762,0.0466,2.8952,2.9294,0.3632,0.1768,0.3106,0.0239
3,visual_baseline_fixed_age_split_seed2025,2025,7,14,0.2780,0.3706,0.2235,2.9482,2.9216,0.3333,0.0924,0.2803,0.0064,0.2836,0.3555,0.1484,2.8952,2.8643,0.3373,0.1451,0.2859,0.0069
4,visual_baseline_fixed_age_split_seed2026,2026,5,12,0.2797,0.3786,0.1896,2.9482,2.9324,0.3345,0.0876,0.2870,0.0197,0.2895,0.3697,0.0788,2.8952,2.8488,0.3472,0.1599,0.2970,0.0219
5,visual_baseline_fixed_age_split_seed2027,2027,6,13,0.2818,0.3754,0.2034,2.9482,2.9541,0.3323,0.0793,0.2868,0.0136,0.2923,0.3710,0.0725,2.8952,2.9087,0.3415,0.1406,0.2966,0.0126
6,visual_baseline_fixed_age_split_seed2048,2048,7,14,0.2937,0.3819,0.1754,2.9482,2.9105,0.3358,0.0860,0.3015,0.0211,0.2763,0.3576,0.1385,2.8952,2.8570,0.3269,0.1432,0.2903,0.0404
7,visual_baseline_fixed_age_split_seed4096,4096,5,12,0.2904,0.3804,0.1819,2.9482,2.8997,0.3442,0.0979,0.2978,0.0200,0.2874,0.3681,0.0870,2.8952,2.8595,0.3416,0.1451,0.2978,0.0299
8,visual_baseline_fixed_age_split_seed7000,7000,6,13,0.2902,0.3778,0.1930,2.9482,2.9347,0.3651,0.1168,0.2980,0.0213,0.2880,0.3723,0.0658,2.8952,2.8631,0.3500,0.1537,0.2955,0.0216
9,visual_baseline_fixed_age_split_seed8192,8192,6,13,0.2821,0.3793,0.1868,2.9482,2.9741,0.3466,0.1203,0.2898,0.0122,0.2953,0.3723,0.0662,2.8952,2.9331,0.3491,0.1638,0.3027,0.0215


,metric,mean,std,min,median,max
0,test_mae,0.2866,0.0071,0.2745,0.2877,0.2953
1,test_rmse,0.3660,0.0076,0.3555,0.3689,0.3762
2,test_r2,0.0968,0.0375,0.0466,0.0829,0.1484
3,test_age_worst_group_mae,0.3407,0.0145,0.3092,0.3416,0.3632
4,test_age_gap,0.1471,0.0187,0.1076,0.1451,0.1768
5,test_gender_worst_group_mae,0.2948,0.0085,0.2802,0.2961,0.3106
6,test_gender_gap,0.0217,0.0091,0.0069,0.0218,0.0404


In [23]:
mean_baseline_predictions = ev.make_mean_baseline_predictions(train_df, test_df)
mean_baseline_metrics = ev.compute_prediction_metrics(mean_baseline_predictions)

ensemble_frame = next(iter(test_predictions.values())).copy()
ensemble_frame["pred"] = np.mean([frame["pred"].to_numpy(dtype=float) for frame in test_predictions.values()], axis=0)
ensemble_metrics = ev.compute_prediction_metrics(ensemble_frame)
_, age_worst, age_gap = ev.compute_group_mae(ensemble_frame, "age_group")
_, gender_worst, gender_gap = ev.compute_group_mae(ensemble_frame, "gender")
ensemble_metrics.update({
    "age_worst_group_mae": age_worst, "age_gap": age_gap,
    "gender_worst_group_mae": gender_worst, "gender_gap": gender_gap,
})
reference_table = pd.DataFrame([
    {"model": "train-mean constant baseline", **mean_baseline_metrics},
    {"model": "ten-seed prediction ensemble (diagnostic)", **ensemble_metrics},
])
display(reference_table.round(4))


,model,n_samples,mae,rmse,r2,tolerant_accuracy,tolerance,one_off_accuracy,binary_threshold,binary_accuracy,binary_f1,binary_roc_auc,binary_confusion_matrix,true_mean,pred_mean,age_worst_group_mae,age_gap,gender_worst_group_mae,gender_gap
0,train-mean constant baseline,15636,0.3124,0.3927,-0.0391,0.2227,0.15,1.0000,3.0,0.5395,0.000,0.5000,"[[8436, 0], [7200, 0]]",2.8952,2.9714,NaN,NaN,NaN,NaN
1,ten-seed prediction ensemble (diagnostic),15636,0.2779,0.3540,0.1557,0.3510,0.15,0.9999,3.0,0.6628,0.518,0.6739,"[[7531, 905], [4367, 2833]]",2.8952,2.8830,0.3351,0.1512,0.2831,0.0151


In [24]:
BOOTSTRAP_RUNS = 1000
bootstrap_summary, _bootstrap_samples = ev.bootstrap_table(
    test_predictions, cluster_col="subject_id", n_boot=BOOTSTRAP_RUNS, seed=SEED
)

subject_rows = []
for run_name, frame in test_predictions.items():
    table = (
        frame.assign(abs_error=np.abs(frame["true"].astype(float) - frame["pred"].astype(float)))
        .groupby("subject_id", observed=True)
        .agg(samples=("abs_error", "size"), mae=("abs_error", "mean"), true_mean=("true", "mean"), pred_mean=("pred", "mean"))
        .reset_index()
    )
    table.insert(0, "run_name", run_name)
    subject_rows.append(table)

subject_seed_metrics = pd.concat(subject_rows, ignore_index=True)
subject_stability = (
    subject_seed_metrics.groupby("subject_id", observed=True)
    .agg(
        samples=("samples", "first"), true_mean=("true_mean", "first"),
        mean_mae=("mae", "mean"), sd_mae=("mae", "std"),
        min_mae=("mae", "min"), max_mae=("mae", "max"),
    )
    .reset_index()
    .sort_values("mean_mae", ascending=False)
)
display(bootstrap_summary.round(4))
display(subject_stability.round(4))


,model,metric,mean,ci_low,ci_high
0,visual_baseline_fixed_age_split_seed42,mae,0.2828,0.2472,0.3255
1,visual_baseline_fixed_age_split_seed42,rmse,0.3590,0.3170,0.4071
2,visual_baseline_fixed_age_split_seed42,r2,0.0915,-0.1577,0.2994
3,visual_baseline_fixed_age_split_seed42,gender_gap,0.0357,0.0000,0.1068
4,visual_baseline_fixed_age_split_seed42,gender_worst_group_mae,0.2970,0.2549,0.3508
...,...,...,...,...,...
65,visual_baseline_fixed_age_split_seed8192,r2,0.0370,-0.2256,0.2366
66,visual_baseline_fixed_age_split_seed8192,gender_gap,0.0392,0.0002,0.1279
67,visual_baseline_fixed_age_split_seed8192,gender_worst_group_mae,0.3094,0.2695,0.3427
68,visual_baseline_fixed_age_split_seed8192,age_gap,0.1383,0.0748,0.1847


,subject_id,samples,true_mean,mean_mae,sd_mae,min_mae,max_mae
8,group03_subject_11,2523,3.1625,0.3747,0.0261,0.3376,0.4285
2,group01_subject_15,1116,2.6351,0.3209,0.0274,0.2764,0.3652
7,group02_subject_20,2459,2.7932,0.3149,0.0341,0.2761,0.3903
5,group02_subject_10,1320,2.8854,0.3021,0.0153,0.2821,0.3284
6,group02_subject_16,2090,2.8524,0.2636,0.0196,0.2439,0.3066
1,group01_subject_10,1610,2.8425,0.2544,0.0248,0.2326,0.3168
4,group02_subject_07,1050,3.0031,0.2517,0.0205,0.2251,0.2833
3,group01_subject_18,1855,2.8776,0.2517,0.0117,0.2312,0.2617
0,group01_subject_08,1613,2.8786,0.1936,0.0078,0.1836,0.2061


In [25]:
# Synthetic visual-missingness stress test.
# Natural missing images remain missing. Additional available images are hidden
# deterministically until each eligible 10-second window reaches the requested
# total missing-image count.
import hashlib

SYNTHETIC_MASK_LEVELS = [4, 6, 8]
SYNTHETIC_MASK_SEED = 20260612
SYNTHETIC_BOOTSTRAP_RUNS = 1000


def make_synthetic_masked_sequences(
    sequence_df,
    visual_missing_col,
    target_missing,
    mask_seed=SYNTHETIC_MASK_SEED,
):
    masked_rows = []

    for _, row in sequence_df.iterrows():
        flags = np.asarray(row[visual_missing_col], dtype=np.float32).copy()
        feature_rows = np.asarray(row["feature_rows"], dtype=np.int64).copy()
        natural_missing = int(flags.sum())

        # Do not unmask naturally missing images. Windows already above the
        # requested level are excluded from that exact-level condition.
        if natural_missing > target_missing:
            continue

        additional_missing = target_missing - natural_missing
        candidates = np.flatnonzero((flags == 0) & (feature_rows >= 0))
        if len(candidates) < additional_missing:
            continue

        key = (
            f"{row['subject_experiment_id']}|{int(row['time_sec'])}|"
            f"{target_missing}|{mask_seed}"
        ).encode("utf-8")
        stable_seed = int.from_bytes(hashlib.sha256(key).digest()[:8], "little")
        rng = np.random.default_rng(stable_seed)
        chosen = rng.choice(candidates, size=additional_missing, replace=False)

        flags[chosen] = 1.0
        feature_rows[chosen] = -1

        updated = row.copy()
        updated[visual_missing_col] = flags.tolist()
        updated["feature_rows"] = feature_rows.tolist()
        updated["natural_visual_missing_count"] = natural_missing
        updated["synthetic_visual_missing_count"] = int(flags.sum())
        updated["synthetic_mask_level"] = target_missing
        masked_rows.append(updated)

    return pd.DataFrame(masked_rows).reset_index(drop=True)


def synthetic_seed_summary_table(seed_metrics):
    rows = []
    for level, group in seed_metrics.groupby("mask_level", observed=True):
        for metric in ["mae", "rmse", "r2"]:
            values = group[metric].astype(float)
            rows.append({
                "mask_level": int(level),
                "metric": metric,
                "mean": float(values.mean()),
                "std": float(values.std()),
                "min": float(values.min()),
                "max": float(values.max()),
            })
    return pd.DataFrame(rows)


synthetic_masking_predictions = {}
synthetic_masking_records = []
synthetic_masking_ensemble_rows = []
synthetic_masking_bootstrap_rows = []

for mask_level in SYNTHETIC_MASK_LEVELS:
    masked_test_df = make_synthetic_masked_sequences(
        test_df,
        visual_missing_col="missing_flags",
        target_missing=mask_level,
    )
    masked_dataset = TemporalFeatureDataset(masked_test_df, feature_store_path)
    masked_loader = DataLoader(masked_dataset, shuffle=False, **loader_kwargs)
    level_frames = []

    print(
        f"Synthetic missing={mask_level}: {len(masked_test_df)} windows, "
        f"{masked_test_df['subject_id'].nunique()} subjects"
    )

    for run_seed in RUN_SEEDS:
        run_name = f"visual_baseline_fixed_age_split_seed{run_seed}"
        model_path = os.path.join(MODEL_DIR, f"{run_name}.pt")
        model = make_model().to(device)
        model.load_state_dict(torch.load(model_path, map_location=device, weights_only=True))

        metrics, frame = evaluate_loader(model, masked_loader, masked_test_df)
        frame["synthetic_mask_level"] = mask_level
        frame["synthetic_mask_seed"] = SYNTHETIC_MASK_SEED
        prediction_name = f"{run_name}_synthetic_missing{mask_level}"
        synthetic_masking_predictions[prediction_name] = frame
        level_frames.append(frame)
        synthetic_masking_records.append({
            "run_name": run_name,
            "run_seed": run_seed,
            "mask_level": mask_level,
            "n_samples": len(frame),
            "n_subjects": frame["subject_id"].nunique(),
            **metrics,
        })
        ev.save_prediction_frame(
            frame,
            os.path.join(RESULTS_DIR, f"{prediction_name}_predictions.csv"),
        )

    ensemble = level_frames[0].copy()
    ensemble["pred"] = np.mean(
        [frame["pred"].to_numpy(dtype=float) for frame in level_frames],
        axis=0,
    )
    ensemble_metrics = ev.compute_prediction_metrics(ensemble)
    synthetic_masking_ensemble_rows.append({
        "mask_level": mask_level,
        "n_subjects": ensemble["subject_id"].nunique(),
        **ensemble_metrics,
    })
    ev.save_prediction_frame(
        ensemble,
        os.path.join(
            RESULTS_DIR,
            f"synthetic_missing{mask_level}_seed_ensemble_predictions.csv",
        ),
    )

    bootstrap_summary_level, _ = ev.cluster_bootstrap_ci(
        ensemble,
        cluster_col="subject_id",
        n_boot=SYNTHETIC_BOOTSTRAP_RUNS,
        seed=SYNTHETIC_MASK_SEED + mask_level,
    )
    bootstrap_summary_level.insert(0, "mask_level", mask_level)
    synthetic_masking_bootstrap_rows.append(bootstrap_summary_level.reset_index(names="metric"))

synthetic_masking_seed_metrics = pd.DataFrame(synthetic_masking_records)
synthetic_masking_seed_summary = synthetic_seed_summary_table(synthetic_masking_seed_metrics)
synthetic_masking_ensemble_metrics = pd.DataFrame(synthetic_masking_ensemble_rows)
synthetic_masking_ensemble_bootstrap = pd.concat(
    synthetic_masking_bootstrap_rows,
    ignore_index=True,
)

display(synthetic_masking_seed_summary.round(4))
display(synthetic_masking_ensemble_metrics.round(4))
display(synthetic_masking_ensemble_bootstrap.round(4))


Synthetic missing=4: 15526 windows, 9 subjects


Synthetic missing=6: 15600 windows, 9 subjects


Synthetic missing=8: 15627 windows, 9 subjects


,mask_level,metric,mean,std,min,max
0,4,mae,0.2913,0.0071,0.2788,0.3020
1,4,rmse,0.3730,0.0080,0.3598,0.3830
2,4,r2,0.0545,0.0406,0.0036,0.1204
3,6,mae,0.3002,0.0102,0.2838,0.3233
4,6,rmse,0.3858,0.0125,0.3664,0.4114
5,6,r2,-0.0071,0.0656,-0.1438,0.0924
6,8,mae,0.3142,0.0228,0.2902,0.3706
7,8,rmse,0.4039,0.0277,0.3742,0.4679
8,8,r2,-0.1049,0.1568,-0.4766,0.0557


,mask_level,n_subjects,n_samples,mae,rmse,r2,tolerant_accuracy,tolerance,one_off_accuracy,binary_threshold,binary_accuracy,binary_f1,binary_roc_auc,binary_confusion_matrix,true_mean,pred_mean
0,4,9,15526,0.2815,0.3590,0.1243,0.3418,0.15,0.9998,3.0,0.6491,0.4703,0.6608,"[[7659, 763], [4685, 2419]]",2.8925,2.8529
1,6,9,15600,0.2879,0.3699,0.0753,0.3379,0.15,0.9997,3.0,0.6279,0.4133,0.6440,"[[7752, 683], [5121, 2044]]",2.8941,2.8329
2,8,9,15627,0.2982,0.3850,0.0005,0.3290,0.15,0.9999,3.0,0.5886,0.3023,0.6145,"[[7805, 631], [5798, 1393]]",2.8949,2.8051


,metric,mask_level,mean,ci_low,ci_high
0,mae,4,0.2808,0.2426,0.3249
1,rmse,4,0.3574,0.3089,0.4096
2,r2,4,0.0962,-0.1442,0.2739
3,gender_gap,4,0.0370,0.0002,0.1097
4,gender_worst_group_mae,4,0.2961,0.2492,0.3461
5,age_gap,4,0.1289,0.0657,0.1926
6,age_worst_group_mae,4,0.3390,0.3076,0.3867
7,mae,6,0.2860,0.2443,0.3330
8,rmse,6,0.3666,0.3148,0.4217
9,r2,6,0.0552,-0.2013,0.2464


In [26]:
split_demographics.to_csv(os.path.join(RESULTS_DIR, "split_demographics.csv"), index=False)
seed_results.to_csv(os.path.join(RESULTS_DIR, "seed_results.csv"), index=False)
seed_summary.to_csv(os.path.join(RESULTS_DIR, "seed_summary.csv"), index=False)
reference_table.to_csv(os.path.join(RESULTS_DIR, "reference_baselines.csv"), index=False)
bootstrap_summary.to_csv(os.path.join(RESULTS_DIR, "subject_bootstrap_summary.csv"), index=False)
subject_seed_metrics.to_csv(os.path.join(RESULTS_DIR, "subject_seed_metrics.csv"), index=False)
subject_stability.to_csv(os.path.join(RESULTS_DIR, "subject_stability.csv"), index=False)
ev.save_prediction_frame(ensemble_frame, os.path.join(RESULTS_DIR, "seed_ensemble_test_predictions.csv"))
synthetic_masking_seed_metrics.to_csv(os.path.join(RESULTS_DIR, "synthetic_masking_seed_metrics.csv"), index=False)
synthetic_masking_seed_summary.to_csv(os.path.join(RESULTS_DIR, "synthetic_masking_seed_summary.csv"), index=False)
synthetic_masking_ensemble_metrics.to_csv(os.path.join(RESULTS_DIR, "synthetic_masking_ensemble_metrics.csv"), index=False)
synthetic_masking_ensemble_bootstrap.to_csv(os.path.join(RESULTS_DIR, "synthetic_masking_ensemble_bootstrap.csv"), index=False)

def json_safe(value):
    """Recursively convert pandas/NumPy objects and non-JSON dictionary keys."""
    if isinstance(value, dict):
        return {str(key): json_safe(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(item) for item in value]
    if isinstance(value, np.generic):
        return value.item()
    return value


with open(os.path.join(RESULTS_DIR, "run_manifest.json"), "w") as file:
    json.dump(
        json_safe({
            "experiment": "Temporal Visual fixed constrained split baseline stability",
            "split_stratify": STRATIFY_COLUMN,
            "split_random_state": SPLIT_RANDOM_STATE,
            "split_constraint_penalty": split_penalty,
            "split_balance_score": split_balance_score,
            "run_seeds": RUN_SEEDS,
            "loss_type": LOSS_TYPE,
            "early_stopping_metric": EARLY_STOPPING_METRIC,
            "mean_baseline_metrics": mean_baseline_metrics,
            "ensemble_metrics": ensemble_metrics,
            "runs": run_records,
        }),
        file,
        indent=2,
        default=str,
    )
print(f"Saved visual baseline stability results to: {RESULTS_DIR}")


Saved visual baseline stability results to: results/Temporal Visual Baseline Stability146


### Reading the results

Use `seed_summary.csv` to compare the visual-only model with the Fusion baseline
on the exact same subject split. A useful model should consistently beat the
train-mean constant predictor across seeds, not only in its best run.
